## Business Problem

The objective of this project is to analyze Coca-Cola’s historical stock performance
to understand long-term growth, volatility, and risk patterns.
This analysis supports investment decision-making and market trend evaluation.

## Tools & Technologies

- SQL (SQLite): Data storage and querying
- Python: Data cleaning, EDA, statistical analysis
- Pandas, Matplotlib, Seaborn
- Power BI: Dashboard visualization

In [3]:
import sqlite3

In [4]:
conn = sqlite3.connect("coca_cola.db")
cursor = conn.cursor()

print("Database Connected!")


Database Connected!


In [5]:
#create table

cursor.execute("""
CREATE TABLE IF NOT EXISTS coca_cola_stock (
    Date TEXT,
    Open REAL,
    High REAL,
    Low REAL,
    Close REAL,
    Adj_Close REAL,
    Volume INTEGER
)
""")

conn.commit()

print("Table Created Successfully!")

Table Created Successfully!


In [7]:
#Basic data checks
#cursor.execute() → runs SQL inside Python
cursor.execute("SELECT COUNT(*) FROM coca_cola_stock")

result = cursor.fetchone()
#fetchone() → gets one result
#result[0] → extracts the number

print("Total Rows:", result[0])

Total Rows: 0


In [17]:
#Insert CSV into SQL Table

import pandas as pd
import os
base_path =os.getcwd()
file_path =os.path.join(base_path ,"KO_CocaCola_Stock_Prices_1980_2026.csv")
df = pd.read_csv(file_path)

df.to_sql("coca_cola_stock", conn, if_exists="replace", index=False)

print("Data Inserted Successfully!")

Data Inserted Successfully!


In [18]:
#BASIC SQL ANALYSIS:

#1. Total Number of Rows
query = "SELECT COUNT(*) AS total_rows FROM coca_cola_stock"
pd.read_sql(query, conn)

,total_rows
0,11615


In [20]:
#2. First 5 Records
query = "SELECT * FROM coca_cola_stock LIMIT 5"
pd.read_sql(query, conn)

,Date,Open,High,Low,Close,Volume,Daily_Return_Pct,Daily_Range,MA_20,MA_50,...,BB_Upper,BB_Lower,BB_Width,RSI_14,Volume_MA_20,Cumulative_Return_Pct,Year,Month,Quarter,Day_of_Week
0,1980-01-02,0.1937,0.1944,0.1895,0.1895,3451200,NaN,0.0049,None,None,...,None,None,None,None,None,0.00,1980,1,1,Wednesday
1,1980-01-03,0.1895,0.1951,0.1881,0.1944,3960000,2.5858,0.0070,None,None,...,None,None,None,None,None,2.59,1980,1,1,Thursday
2,1980-01-04,0.1944,0.1965,0.1937,0.1965,1694400,1.0802,0.0028,None,None,...,None,None,None,None,None,3.69,1980,1,1,Friday
3,1980-01-07,0.1965,0.1972,0.1951,0.1958,4396800,-0.3562,0.0021,None,None,...,None,None,None,None,None,3.32,1980,1,1,Monday
4,1980-01-08,0.1958,0.1979,0.1958,0.1972,3244800,0.7150,0.0021,None,None,...,None,None,None,None,None,4.06,1980,1,1,Tuesday


In [21]:
#3. Check Minimum & Maximum Closing Price

query = """
SELECT 
    MIN(Close) AS Lowest_Close,
    MAX(Close) AS Highest_Close
FROM coca_cola_stock
"""
pd.read_sql(query, conn)

,Lowest_Close,Highest_Close
0,0.1651,74.81


In [22]:
#4. Average Closing Price 
query = """
SELECT AVG(Close) AS Average_Close
FROM coca_cola_stock
"""
pd.read_sql(query, conn)

,Average_Close
0,18.375667


In [23]:
#5. Highest Trading Volume Ever 
query = """
SELECT Date, Volume
FROM coca_cola_stock
ORDER BY Volume DESC
LIMIT 1
"""
pd.read_sql(query, conn)

,Date,Volume
0,2009-09-18,124169000


In [24]:
#YEARLY ANALYSIS:
#6. Yearly Average Closing Price
query = """
SELECT 
    strftime('%Y', Date) AS Year,
    AVG(Close) AS Avg_Close
FROM coca_cola_stock
GROUP BY Year
ORDER BY Year
"""
pd.read_sql(query, conn)


,Year,Avg_Close
0,1980,0.191334
1,1981,0.215474
2,1982,0.252988
3,1983,0.364341
4,1984,0.429171
5,1985,0.543952
6,1986,0.859684
7,1987,1.081064
8,1988,1.006697
9,1989,1.574516


In [25]:
#7. Yearly Highest & Lowest Price

query = """
SELECT 
    strftime('%Y', Date) AS Year,
    MAX(High) AS Yearly_High,
    MIN(Low) AS Yearly_Low
FROM coca_cola_stock
GROUP BY Year
ORDER BY Year
"""
pd.read_sql(query, conn)

,Year,Yearly_High,Yearly_Low
0,1980,0.2229,0.1628
1,1981,0.2493,0.1925
2,1982,0.3680,0.1909
3,1983,0.4157,0.3122
4,1984,0.4949,0.3542
5,1985,0.6976,0.4511
6,1986,1.0801,0.6067
7,1987,1.3136,0.7212
8,1988,1.1676,0.8767
9,1989,2.1375,1.1192


In [26]:
#VOLATILITY ANALYSIS:

#8. Daily Price Range
query = """
SELECT 
    Date,
    (High - Low) AS Daily_Range
FROM coca_cola_stock
ORDER BY Daily_Range DESC
LIMIT 5
"""
pd.read_sql(query, conn)

,Date,Daily_Range
0,2020-03-16,5.6309
1,2022-05-18,4.0778
2,2020-03-20,3.8266
3,2025-04-07,3.2584
4,2025-04-04,3.1508


In [27]:
#DECADE COMPARISON :
#9. Compare 1990s vs 2010s Average Close

query = """
SELECT 
    CASE 
        WHEN strftime('%Y', Date) BETWEEN '1990' AND '1999' THEN '1990s'
        WHEN strftime('%Y', Date) BETWEEN '2010' AND '2019' THEN '2010s'
    END AS Decade,
    AVG(Close) AS Avg_Close
FROM coca_cola_stock
WHERE strftime('%Y', Date) BETWEEN '1990' AND '1999'
   OR strftime('%Y', Date) BETWEEN '2010' AND '2019'
GROUP BY Decade
"""
pd.read_sql(query, conn)

,Decade,Avg_Close
0,1990s,8.450666
1,2010s,29.169123
